# Fuzija pristupa (stacking nad OOF predikcijama)

Svaki osnovni pristup vidi drugačiji signal, pa kombinacija njihovih predikcija treba da
bude bolja od bilo kog pojedinačnog. Ovde to radimo stacking-om: meta-model (Ridge u log
prostoru) uči kako da pomeša OOF predikcije osnovnih modela. To je ujedno i jeftin
benchmark za zajednički multimodalni model iz `04` (dva načina da se isti pristupi spoje).
Pošteno je jer je svaka OOF predikcija napravljena modelom koji celu opštinu tog naselja
nije video u treningu, a meta-model se ocenjuje istom GroupKFold podelom po opštinama
(fituje se na trening opštinama, ocenjuje na validacionim).

Usput i post-hoc kalibracija po pristupu: stacking sa jednim ulazom je tačno regresija
nagib+presek na log-log skali, što ispravlja sistematsku kompresiju predikcija ka sredini
(dijagnostikovanu preko `oof_kalib_nagib` metrike). Na kraju tabela: sirovo vs kalibrisano
po pristupu vs fuzija.

Ulaz su `oof_<pristup>.parquet` fajlovi koje trenirački notebooci snimaju u OUT_DIR; ovaj
notebook ne trenira mreže i ne zahteva GPU.

## Instalacija

In [ ]:
%pip install -q "mlflow>=3.0" scikit-learn

## Konfiguracija

In [ ]:
import os
import sys
# koren repoa na sys.path (penji se od radnog dir dok ne nadje core/)
koren = os.getcwd()
while not os.path.isdir(os.path.join(koren, "core")) and os.path.dirname(koren) != koren:
    koren = os.path.dirname(koren)
sys.path.insert(0, koren)

import numpy as np, pandas as pd
import mlflow
from sklearn.linear_model import Ridge
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
from core import (
    seed_everything,
    make_folds, oof_metrics, run_metrics, summary_line,
    cv_summary_figure, calibration_figure, GLAVNE_KOLONE,
    setup_mlflow, output_dir, save_oof,
)

OUT_DIR = output_dir()   # tu su oof_<pristup>.parquet iz treniracih notebooka
CFG = {
    "alpha": 1.0,     # Ridge regularizacija (mala; ulaza je 1-5, uzoraka ~4.6k)
    "seed": 42,
}
seed_everything(CFG["seed"])

# kandidati za ulaz u fuziju: koristi se svaki za koji postoji OOF parquet
# GBM i TabFM nisu kandidati: nad istim atributima su kao MLP, pa bi ulazi bili
# gotovo kolinearni; oba ostaju referentne linije u 03, van fuzije
KANDIDATI = ["tiles_log", "tiles_linear", "footprint", "footprint_tab", "multimodal"]

## Učitavanje OOF predikcija osnovnih modela

In [ ]:
osnove = {}
for ime in KANDIDATI:
    put = f"{OUT_DIR}/oof_{ime}.parquet"
    if os.path.exists(put):
        osnove[ime] = pd.read_parquet(put)
        print(f"  {ime:18s} {len(osnove[ime])} naselja")
    else:
        print(f"  {ime:18s} nema ({put})")
PRISTUPI = list(osnove)
if len(PRISTUPI) < 2:
    raise RuntimeError("za fuziju trebaju bar dva OOF parqueta (pokreni trenirace notebooke)")

# spajanje na preseku naselja; pop i opstina iz prvog (identicni su u svim fajlovima)
prvi = PRISTUPI[0]
df = osnove[prvi].rename(columns={"pred": f"pred_{prvi}"})
for ime in PRISTUPI[1:]:
    t = osnove[ime].rename(columns={"pred": f"pred_{ime}"})
    df = df.merge(t[["naselje_maticni_broj", f"pred_{ime}"]], on="naselje_maticni_broj", how="inner")
print(f"presek: {len(df)} naselja | pristupi: {PRISTUPI}")

# ulazi meta-modela su log1p predikcija (isti prostor kao cilj), cilj log1p(pop)
for ime in PRISTUPI:
    df[f"x_{ime}"] = np.log1p(df[f"pred_{ime}"].clip(lower=0)).astype("float32")
df["ylog"] = np.log1p(df["pop"]).astype("float32")

FOLDS = make_folds(df)
broj_opstina = df["opstina_maticni_broj"].nunique()
print(f"naselja {len(df)} | opstina {broj_opstina} | foldova {len(FOLDS)}")

## Meta-model (stacking / kalibracija)

In [ ]:
def calibrate_cv(x_kolona, metod):
    # Post-hoc kalibracija jednog pristupa; vraca (oof_pred_pop, fold_r2). Uci monotonu
    # funkciju predvidjeno -> ispravljeno na opstinama van folda, pa popravlja skalu a ne
    # rangiranje. "ridge" je nagib+presek u log-log prostoru, "isotonic" bilo koja monotona
    # kriva (osetljivija na retka velika naselja).
    oof = pd.Series(np.nan, index=df.naselje_maticni_broj.values, dtype="float64")
    fold_r2 = []
    for train_frame, val_frame in FOLDS:
        xtr, ytr = train_frame[[x_kolona]].values, train_frame["ylog"].values
        xva = val_frame[[x_kolona]].values
        if metod == "ridge":
            plog = Ridge(alpha=CFG["alpha"]).fit(xtr, ytr).predict(xva)
        else:
            iso = IsotonicRegression(out_of_bounds="clip", increasing=True)
            plog = iso.fit(xtr.ravel(), ytr).predict(xva.ravel())
        oof.loc[val_frame.naselje_maticni_broj.values] = np.clip(np.expm1(plog), 0, None)
        fold_r2.append(r2_score(val_frame["ylog"].values, plog))
    return oof.loc[df.naselje_maticni_broj.values].values, fold_r2


def stacking_cv(x_kolone):
    # Stacking: Ridge nad vise ulaza, fitovan na trening opstinama. Vraca (oof_pred_pop,
    # fold_r2, koeficijenti); koeficijenti su prosek preko foldova.
    oof = pd.Series(np.nan, index=df.naselje_maticni_broj.values, dtype="float64")
    fold_r2, koef = [], []
    for train_frame, val_frame in FOLDS:
        reg = Ridge(alpha=CFG["alpha"]).fit(train_frame[x_kolone], train_frame["ylog"])
        plog = reg.predict(val_frame[x_kolone])
        oof.loc[val_frame.naselje_maticni_broj.values] = np.clip(np.expm1(plog), 0, None)
        fold_r2.append(r2_score(val_frame["ylog"], plog))
        koef.append(np.r_[reg.coef_, reg.intercept_])
    oof_pred = oof.loc[df.naselje_maticni_broj.values].values
    koef = np.mean(koef, axis=0)
    # float(): np.float32 nije JSON-serijabilan pa bi mlflow.log_dict pukao
    return oof_pred, fold_r2, {k: round(float(v), 3)
                               for k, v in zip([*x_kolone, "presek"], koef)}

## Evaluacija: sirovo vs kalibrisano vs fuzija

In [ ]:
KLJUCNE = GLAVNE_KOLONE

def run():
    # Kalibracija svakog pristupa pa stacking fuzija, u jednom MLflow runu.
    setup_mlflow()   # Databricks workspace; lokalno mlruns/, ili Databricks preko env varijabli
    stvarno = df["pop"].values.astype("float32")
    redovi, kalibrisani = [], {}

    with mlflow.start_run(run_name=f"fuzija-stacking-cv{len(FOLDS)}"):
        mlflow.log_params({**CFG, "pristup": "fuzija_stacking", "meta_model": "Ridge(log1p)",
                           "ulazi": ",".join(PRISTUPI), "cv": f"GroupKFold(opstina) x{len(FOLDS)}",
                           "n_naselja": len(df), "n_opstina": int(broj_opstina)})

        # po pristupu: sirovo, pa obe kalibracije. Isti foldovi za sve, pa su
        # kalibrisani brojevi uporedivi izmedju pristupa.
        for ime in PRISTUPI:
            sirovo = df[f"pred_{ime}"].values
            m_sirovo = oof_metrics(stvarno, sirovo, df)
            redovi.append({"varijanta": f"{ime} (sirovo)", **m_sirovo})
            print(f"[{ime}] sirov nagib {m_sirovo.get('oof_kalib_nagib', float('nan')):.3f}"
                  f" | bias {m_sirovo.get('oof_bias', float('nan')):.2f}")
            for metod in ("ridge", "isotonic"):
                kal_pred, kal_r2 = calibrate_cv(f"x_{ime}", metod)
                redovi.append({"varijanta": f"{ime} (kalibrisano: {metod})",
                               **run_metrics(kal_r2, stvarno, kal_pred, df)})
                kalibrisani[(ime, metod)] = kal_pred

        # fuzija svih pristupa
        fuz_pred, fuz_r2, fuz_koef = stacking_cv([f"x_{ime}" for ime in PRISTUPI])
        print(f"[fuzija] koeficijenti {fuz_koef}")
        agg = run_metrics(fuz_r2, stvarno, fuz_pred, df)
        redovi.append({"varijanta": "fuzija (stacking)", **agg})

        mlflow.log_metrics(agg)
        mlflow.log_dict(fuz_koef, "fuzija_koeficijenti.json")
        put_oof = save_oof(df, fuz_pred, "fuzija", OUT_DIR)
        mlflow.log_artifact(put_oof)
        fig = cv_summary_figure(fuz_r2, agg, stvarno, fuz_pred, df, label="fuzija")
        plt.show()
        mlflow.log_figure(fig, "cv_evaluacija_fuzija.png")

        poredjenje = pd.DataFrame(redovi).set_index("varijanta").reindex(columns=KLJUCNE)
        mlflow.log_text(poredjenje.to_string(), "poredjenje_fuzija.txt")

        # efekat kalibracije po pristupu: nagib pre/posle i sta se desilo sa greskom
        fig_k = calibration_figure(stvarno, df, PRISTUPI, kalibrisani)
        plt.show()
        mlflow.log_figure(fig_k, "kalibracija_po_pristupu.png")

    print(summary_line(agg, "fuzija"))
    return poredjenje


poredjenje = run()
print("\n=== Poredjenje: sirovo vs kalibrisano vs fuzija ===")
display(poredjenje)